<a href="https://colab.research.google.com/github/Yami2912/HER360/blob/feature%2Fhealthcare/HER360-Healthcare-ML/notebooks/fertility/ovulation_predictor/03_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import userdata
import os, pathlib, sys, json
TOKEN = userdata.get("GITHUB_TOKEN")
os.chdir("/content")
if not os.path.exists("/content/HER360"):
    !git clone https://{TOKEN}@github.com/Yami2912/HER360.git
%cd /content/HER360
!git config user.name  "Yami2912"
!git config user.email "yamipatel0618@gmail.com"
!git fetch origin
!git checkout feature/healthcare
!git pull origin feature/healthcare

BASE = pathlib.Path("/content/HER360/HER360-Healthcare-ML")
print((BASE / "datasets/fertility/ovulation_predictor/processed/ovulation_model_table_v1.csv").exists())

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
pd.set_option("display.width", 200)

T = "target_estimated_ovulation_day"
df = pd.read_csv(BASE / "datasets/fertility/ovulation_predictor/processed/ovulation_model_table_v1.csv")

train = df[df.split == "train"].copy()
val   = df[df.split == "validation"].copy()
# NOTE: the test split is intentionally NOT used anywhere in this notebook.
print("train rows:", len(train), "| validation rows:", len(val))
print(val.history_status.value_counts().to_dict())

In [ ]:
def metrics(y, p):
    e = (y - p).abs()
    return {"n": len(y),
            "MAE": round(float(e.mean()), 2),
            "RMSE": round(float(np.sqrt(((y - p) ** 2).mean())), 2),
            "within_1_day_%": round(100 * float((e <= 1).mean())),
            "within_2_days_%": round(100 * float((e <= 2).mean()))}

# Parameters learned from TRAIN only
global_median = float(train[T].median())
has_tr = train.n_prev_cycles >= 1
learned_offset = float((train[has_tr][T] - train[has_tr].prev_cycle_mean).median())
print("global median:", global_median, "| learned offset:", round(learned_offset, 2))

def predict(d, method):
    p = pd.Series(global_median, index=d.index, dtype=float)   # fallback: no history
    has = d.n_prev_cycles >= 1
    if method == "calendar_minus14":
        p[has] = d.prev_cycle_mean[has] - 14          # common rule of thumb: ovulation ~14 days before next period
    elif method == "calendar_learned_offset":
        p[has] = d.prev_cycle_mean[has] + learned_offset
    return p

methods = ["global_median", "calendar_minus14", "calendar_learned_offset"]
rows = []
for m in methods:
    p = predict(val, m)
    rows.append({"method": m, "subset": "all", **metrics(val[T], p)})
    for status, g in val.groupby("history_status"):
        rows.append({"method": m, "subset": status, **metrics(g[T], p[g.index])})

results = pd.DataFrame(rows)
display(results)

In [ ]:
plot = results[results.subset == "all"].set_index("method")["MAE"]
ax = plot.plot.bar(figsize=(6, 3.5), rot=15, edgecolor="black")
ax.set_ylabel("MAE (days) on validation")
ax.set_title("Baseline comparison")
for i, v in enumerate(plot): ax.text(i, v + 0.03, f"{v:.2f}", ha="center")
plt.tight_layout(); plt.show()

In [ ]:
NB_DIR = BASE / "notebooks/fertility/ovulation_predictor"
results.to_csv(NB_DIR / "baseline_results_v1.csv", index=False)
(NB_DIR / "baseline_params_v1.json").write_text(json.dumps({
    "global_median": global_median,
    "learned_offset": learned_offset,
    "fitted_on": "train split only",
    "scored_on": "validation split only",
    "luteal_rule_of_thumb_days": 14,
    "note": "Engineering baseline. Not medical guidance.",
}, indent=2))
print("saved")

In [ ]:
%cd /content/HER360
!git add HER360-Healthcare-ML/notebooks
!git commit -m "Fertility: ovulation baseline results v1"
!git push origin feature/healthcare